In [47]:
# 所需要的所有库

import pandas as pd
import numpy as np
import json
import ast
import io
from PIL import Image
import numpy as np

import torch
from torchvision import models, transforms

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt

# 如需大模型embedding
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

print("草你妈，总算找全了！")

草你妈，总算找全了！


In [48]:
# 倒入训练集和测试集

# 载入训练集
df_train = pd.read_csv("glaucoma_train.csv")

# 载入测试集
df_test = pd.read_csv("glaucoma_test.csv")


In [49]:
# 首先，把训练集的所有列名展示出来

print("训练集列名：", df_train.columns.tolist())

print("\n训练集前5行样本：")
print(df_train.head())

# 草泥马的傻逼项目，数据集都没有

训练集列名： ['image', 'label', 'filename', 'annotation', 'description']

训练集前5行样本：
                                               image  label  \
0  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...      0   
1  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...      1   
2  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...      0   
3  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...      0   
4  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...      0   

                  filename annotation  \
0   acrima_glaucoma_57.png   glaucoma   
1   acrima_normal_1058.png     normal   
2  acrima_glaucoma_188.png   glaucoma   
3  acrima_glaucoma_182.png   glaucoma   
4   acrima_glaucoma_21.png   glaucoma   

                                         description  
0  {\n    "fundus_features": {\n        "optic_di...  
1  {\n    "fundus_features": {\n        "optic_di...  
2  {\n    "fundus_features": {\n        "optic_di...  
3  {\n    "fundus_features": {\n        "optic_di...  
4  {\n    "fundus_fea

In [50]:
# filename是image的名称，label和annotation是同一个东西。
# 0 是正常眼睛，1是傻逼青光眼

for i, desc in enumerate(df_train['description'].head(5)):
    desc_dict = json.loads(desc)
    print(f"\n样本 {i} 的 description 字段：")
    print(json.dumps(desc_dict, indent=2, ensure_ascii=False))

# 老子先展示description的内容看看



样本 0 的 description 字段：
{
  "fundus_features": {
    "optic_disc_size": "large",
    "cup_to_disc_ratio": 0.8,
    "neuroretinal_rim": "thinned in all regions, especially inferior and superior",
    "isnt_rule_followed": false,
    "rim_pallor": true,
    "rim_color": "pale",
    "bayoneting": true,
    "sharp_edge": true,
    "laminar_dot_sign": true,
    "notching": true,
    "rim_thinning": true,
    "additional_observations": null
  },
  "glaucoma_risk_assessment": "high risk",
  "confidence_level": 0.9
}

样本 1 的 description 字段：
{
  "fundus_features": {
    "optic_disc_size": "normal",
    "cup_to_disc_ratio": 0.4,
    "neuroretinal_rim": "The thickness of the neuroretinal rim appears adequate in all regions: superior, inferior, nasal, and temporal.",
    "isnt_rule_followed": true,
    "rim_pallor": false,
    "rim_color": "orange",
    "bayoneting": false,
    "sharp_edge": false,
    "laminar_dot_sign": false,
    "notching": false,
    "rim_thinning": false,
    "additional_obs

In [51]:
# 老子先把人为评估的因素提取出来，操你妈

def extract_human_eval(desc):
    try:
        desc_dict = json.loads(desc)
        return pd.Series({
            'glaucoma_risk_assessment': desc_dict.get('glaucoma_risk_assessment', None),
            'confidence_level': desc_dict.get('confidence_level', None)
        })
    except Exception:
        return pd.Series({'glaucoma_risk_assessment': None, 'confidence_level': None})

human_eval_df = df_train['description'].apply(extract_human_eval)

# 合并回训练集
df_train = pd.concat([df_train, human_eval_df], axis=1)

print(df_train[['glaucoma_risk_assessment', 'confidence_level']].head())

# 分类型 one-hot
X_human_cat = pd.get_dummies(df_train[['glaucoma_risk_assessment']], dummy_na=True)
# 数值型
X_human_num = df_train[['confidence_level']]

  glaucoma_risk_assessment  confidence_level
0                high risk              0.90
1             very healthy              0.95
2                high risk              0.90
3                high risk              0.90
4                high risk              0.90


In [52]:
# 然后老子来处理这些非人为的结构化因素

def extract_structured_features(desc):
    try:
        desc_dict = json.loads(desc)
        ff = desc_dict.get('fundus_features', {})
        # 排除 neuroretinal_rim 和 additional_observations（文本和无用变量）
        keys = [
            "optic_disc_size",
            "cup_to_disc_ratio",
            "isnt_rule_followed",
            "rim_pallor",
            "rim_color",
            "bayoneting",
            "sharp_edge",
            "laminar_dot_sign",
            "notching",
            "rim_thinning"
        ]
        return pd.Series({k: ff.get(k, None) for k in keys})
    except Exception:
        return pd.Series({k: None for k in keys})

structured_df = df_train['description'].apply(extract_structured_features)

# 合并回训练集
df_train = pd.concat([df_train, structured_df], axis=1)

print(structured_df.head())

# 结构化变量列表
structured_cols = [
    'optic_disc_size', 'cup_to_disc_ratio', 'isnt_rule_followed', 'rim_pallor',
    'rim_color', 'bayoneting', 'sharp_edge', 'laminar_dot_sign', 'notching', 'rim_thinning'
]
structured_df = df_train[structured_cols]
X_structured = pd.get_dummies(structured_df, dummy_na=True)

  optic_disc_size  cup_to_disc_ratio  isnt_rule_followed  rim_pallor  \
0           large                0.8               False        True   
1          normal                0.4                True       False   
2          normal                0.7               False        True   
3          normal                0.8               False       False   
4          normal                0.8               False       False   

  rim_color  bayoneting  sharp_edge  laminar_dot_sign  notching  rim_thinning  
0      pale        True        True              True      True          True  
1    orange       False       False             False     False         False  
2      pale       False        True             False     False          True  
3      pink       False        True             False     False          True  
4      pink        True        True             False      True          True  


In [53]:
# neuroretinal_rim这个傻逼变量，是你妈逼的文本描述

def extract_rim_text(desc):
    try:
        desc_dict = json.loads(desc)
        ff = desc_dict.get('fundus_features', {})
        return ff.get('neuroretinal_rim', '')
    except Exception:
        return ''

df_train['neuroretinal_rim_text'] = df_train['description'].apply(extract_rim_text)
print(df_train['neuroretinal_rim_text'].head())

0    thinned in all regions, especially inferior an...
1    The thickness of the neuroretinal rim appears ...
2    thinning observed in the superior and inferior...
3    thinner in the superior and inferior regions, ...
4    thinner in the inferior and superior regions, ...
Name: neuroretinal_rim_text, dtype: object


In [54]:
# 提取 neuroretinal_rim 文本内容

def extract_rim_text(desc):
    try:
        desc_dict = json.loads(desc)
        ff = desc_dict.get('fundus_features', {})
        return ff.get('neuroretinal_rim', '')
    except Exception:
        return ''

df_train['neuroretinal_rim_text'] = df_train['description'].apply(extract_rim_text)

# 用 BERT 转 embedding

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

MODEL_NAME = 'bert-base-uncased'  # 如需医学BERT可换
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def text_to_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=32)
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # 取[CLS]
    return embedding

rim_texts = df_train['neuroretinal_rim_text'].fillna('').astype(str).tolist()
rim_embeddings = []
for txt in tqdm(rim_texts, desc="Encoding neuroretinal_rim"):
    rim_embeddings.append(text_to_embedding(txt, tokenizer, model))
rim_embeddings = np.stack(rim_embeddings)  # shape (N, 768)

Encoding neuroretinal_rim:   0%|          | 0/589 [00:00<?, ?it/s]/opt/anaconda3/envs/xgb/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Encoding neuroretinal_rim: 100%|██████████| 589/589 [00:20<00:00, 28.17it/s]


In [55]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
resnet = models.resnet18(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

def extract_image_feature(img_bytes_dict):
    try:
        img = Image.open(io.BytesIO(img_bytes_dict["bytes"])).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0)
        with torch.no_grad():
            feat = resnet(img_tensor).squeeze().numpy()
        return feat  # shape: (512,)
    except Exception:
        return np.zeros(512)

img_vecs = []
for img_raw in df_train["image"]:
    img_dict = ast.literal_eval(img_raw)
    img_vec = extract_image_feature(img_dict)
    img_vecs.append(img_vec)
img_vecs = np.stack(img_vecs)  # shap

/opt/anaconda3/envs/xgb/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/xgb/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [56]:
# 合并这些傻逼数据

# 全部拼接（注意要 .values 变成 numpy，防止拼接出错）
X_all = np.hstack([
    X_structured.values,
    img_vecs,
    rim_embeddings,
    X_human_cat.values,
    X_human_num.values
])

# 联合 label
y = df_train['label'].values

print(X_all)

[[0.8 False True ... False False 0.9]
 [0.4 True False ... True False 0.95]
 [0.7 False True ... False False 0.9]
 ...
 [0.8 False True ... False False 0.9]
 [0.8 False True ... False False 0.9]
 [0.7 False True ... False False 0.9]]


In [57]:
# 1. 划分训练/验证集
X_train, X_val, y_train, y_val = train_test_split(X_all, y, test_size=0.5, random_state=42)

# 2. 训练 XGBoost
model = xgb.XGBClassifier(
    objective='binary:logistic',    # 二分类任务
    tree_method='hist',             # 高效直方图算法
    use_label_encoder=False,        # 新版本不使用 label 编码器
    eval_metric='logloss',          # 评价指标：对数损失
    n_estimators=500,               # 树的数量，更多树可能更准
    max_depth=6,                    # 树最大深度，防止过拟合
    learning_rate=0.05,             # 学习率（eta），降低收敛速度，提升泛化
    subsample=0.8,                  # 每棵树用80%样本，减少过拟合
    colsample_bytree=0.8,           # 每棵树用80%特征
    min_child_weight=1,             # 叶子节点最小权重，防止过拟合
    gamma=0,                        # 树分裂时的最小损失减少量
    reg_alpha=0,                    # L1正则化
    reg_lambda=1,                   # L2正则化
    scale_pos_weight=1,             # 类别不平衡时调整
    random_state=42,
    verbosity=1   
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

# 3. 评估
y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f"\nValidation Accuracy: {acc:.4f}")
print("\n分类报告：")
print(classification_report(y_val, y_pred))


[0]	validation_0-logloss:0.61337
[1]	validation_0-logloss:0.57059
[2]	validation_0-logloss:0.53221
[3]	validation_0-logloss:0.49726
[4]	validation_0-logloss:0.46542
[5]	validation_0-logloss:0.43627
[6]	validation_0-logloss:0.40954
[7]	validation_0-logloss:0.38494
[8]	validation_0-logloss:0.36221
[9]	validation_0-logloss:0.34112
[10]	validation_0-logloss:0.32155
[11]	validation_0-logloss:0.30341
[12]	validation_0-logloss:0.28650
[13]	validation_0-logloss:0.27073
[14]	validation_0-logloss:0.25601
[15]	validation_0-logloss:0.24220
[16]	validation_0-logloss:0.22929
[17]	validation_0-logloss:0.21719
[18]	validation_0-logloss:0.20581
[19]	validation_0-logloss:0.19514
[20]	validation_0-logloss:0.18507
[21]	validation_0-logloss:0.17562
[22]	validation_0-logloss:0.16670
[23]	validation_0-logloss:0.15831
[24]	validation_0-logloss:0.15039
[25]	validation_0-logloss:0.14293
[26]	validation_0-logloss:0.13589
[27]	validation_0-logloss:0.12922
[28]	validation_0-logloss:0.12293
[29]	validation_0-loglos

/opt/anaconda3/envs/xgb/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [02:20:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[42]	validation_0-logloss:0.06328
[43]	validation_0-logloss:0.06051
[44]	validation_0-logloss:0.05787
[45]	validation_0-logloss:0.05536
[46]	validation_0-logloss:0.05296
[47]	validation_0-logloss:0.05070
[48]	validation_0-logloss:0.04854
[49]	validation_0-logloss:0.04648
[50]	validation_0-logloss:0.04453
[51]	validation_0-logloss:0.04266
[52]	validation_0-logloss:0.04091
[53]	validation_0-logloss:0.03923
[54]	validation_0-logloss:0.03764
[55]	validation_0-logloss:0.03613
[56]	validation_0-logloss:0.03471
[57]	validation_0-logloss:0.03333
[58]	validation_0-logloss:0.03202
[59]	validation_0-logloss:0.03076
[60]	validation_0-logloss:0.02957
[61]	validation_0-logloss:0.02843
[62]	validation_0-logloss:0.02735
[63]	validation_0-logloss:0.02633
[64]	validation_0-logloss:0.02534
[65]	validation_0-logloss:0.02441
[66]	validation_0-logloss:0.02352
[67]	validation_0-logloss:0.02266
[68]	validation_0-logloss:0.02184
[69]	validation_0-logloss:0.02107
[70]	validation_0-logloss:0.02033
[71]	validatio

In [58]:
feature_names = (
    list(X_structured.columns)
    + [f'img_{i}' for i in range(img_vecs.shape[1])]
    + [f'rim_emb_{i}' for i in range(rim_embeddings.shape[1])]
    + list(X_human_cat.columns)
    + list(X_human_num.columns)
)

importances = model.feature_importances_

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("\n影响最大的前五个特征：")
print(imp_df.head(5))


影响最大的前五个特征：
                                    feature  importance
1                        isnt_rule_followed    0.518719
0                         cup_to_disc_ratio    0.244133
7                              rim_thinning    0.212270
1300  glaucoma_risk_assessment_very healthy    0.008833
1007                            rim_emb_478    0.008114
